In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/uciml/pima-indians-diabetes-database/diabetes.csv


# Importação do dataset: Carregamento dos Dados



In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


df = pd.read_csv("/kaggle/input/datasets/organizations/uciml/pima-indians-diabetes-database/diabetes.csv")


df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### **O que foi feito?**

O projeto foi iniciado com a importação da biblioteca Pandas, Numpy, SVC, StandardScaler e Accuracy.

E para validar se a importação não corrompeu os dados, e que as colunas (características) e rótulos (doenças) foram importados sem erros, utilizamos o 'Head' para exibir as 5 primeiras linhas.

# Introdução ao Tratamento de Dados

In [3]:
colunas_para_tratar = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in colunas_para_tratar:
    mediana = df[df[col]!= 0 ][col].median()
    df[col] = df[col].replace(0, mediana)


### **O que foi feito?**

Ao explorar o dataset, foi notado que colunas como **Glicose** (Glucose), **Pressão** (BloodPressure) e **IMC** (BMI) possuem valores iguais a `0`. Numericamente isso é possível, mas clinicamente é um erro de coleta (seriam dados faltantes).

**Estratégia:** Substituimos esses zeros pela **Mediana** de cada coluna.

# Separação de Variáveis e Divisão em Treino e Teste



In [4]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### **O que foi feito?**

Com os dados limpos, foi preparado o ambiente para o aprendizado supervisionado. 

1. **Separação de X e y:** Isolamos a coluna alvo (`Outcome`) das características.
2. **Divisão Treino/Teste:** Tiramos a amostra de **20%** dos dados para teste e **80%** para o treinamento.
   O `random_state=42` garante que essa divisão seja sempre a mesma toda vez que    rodarmos o código.

# Modelo 1: SVM sem Normalização

In [5]:
svc_sem_normalizar = SVC(kernel='linear')
svc_sem_normalizar.fit(X_train, y_train)



SVC(kernel='linear')

### **O que foi feito?**


Nesta etapa, treinaremos o **Support Vector Machine (SVM)** utilizando um `kernel='linear'`. 
O objetivo é ver como o SVM se vira com os dados brutos, onde as escalas estão todas bagunçadas: como a **Glicose** batendo na casa dos 100 ou 200, enquanto o **Pedigree de Diabetes** é um número decimal minúsculo.

* **Kernel:** Linear (uma linha reta tentando separar quem tem e quem não tem diabetes).
* **Dados:** Brutos (apenas com o tratamento de medianas realizado anteriormente)

##   Acurácia "Bruta"



In [6]:
acuracia_sem_normalizar = accuracy_score(y_test, svc_sem_normalizar.predict(X_test))

print('Acurácia sem normalização dos dados {:.2%}'.format(acuracia_sem_normalizar))

Acurácia sem normalização dos dados 75.97%


### **O que foi feito?**

Foi feito o cálculo da **Acurácia** do SVM treiando sobre os dados "brutos", esse valor representa a precisão do SVM tentando "adivinhar" o diagnóstico apenas com os dados originais.



# Modelo 2: SVM Com StandardScaler

Instanciação da classe `StandardScaler`

In [7]:
scaler = StandardScaler()

O `StandardScaler` é essencial para o **SVM**, pois evita que o cálculo do hiperplano seja distorcido por variáveis com grandes amplitudes térmicas ou numéricas, como a Glicose.

##  Ajuste e Transformação dos Dados

In [8]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### **O que foi feito?**


1. **Ajuste e Treino (`fit_transform`):** O computador olha para os dados de treino, descobre a média e o desvio padrão de cada coluna e já aplica a escala de uma vez.
2. **Transformação do Teste (`transform`):** Usamos a "régua" que o modelo aprendeu no treino para transformar os dados de teste.


Com isso, todos os nossos dados (Glicose, IMC, Idade, etc.) estão na mesma escala, agora o SVM não ira se confundir com os números grandes.

##  Treinamento do Modelo com Dados Padronizados

In [9]:
svm_scaled = SVC(kernel='linear')
svm_scaled.fit(X_train_scaled, y_train)

SVC(kernel='linear')

### **O que foi feito?**


Com as variáveis na mesma escala, continuamos com o treinamento de um novo **SVM (Kernel Linear)**. 

A expectativa é que, com a eliminação dos números grandes entre atributos como Glicose e IMC, o algoritmo consiga definir um hiperplano, de um jeito justo.

## Avaliação de Desempenho do Modelo Padronizado

In [10]:
acuracia_scaled = accuracy_score(y_test, svm_scaled.predict(X_test_scaled))
print("Acurácia com StandardScaler: {:.2%}".format(acuracia_scaled))

Acurácia com StandardScaler: 75.32%


### **O que foi feito?**


Foi feito o cálculo da Acurácia, utilizando o conjunto de teste com os dados normalizados.

O objetivo é ver **ganho de performance**. Se a acurácia subiu, provamos que o SVM realmente precisava dos dados normalizados para um hiperplano de separação sem se perder nas escalas diferentes de cada coluna.

## Análise Comparativa e Validação da Hipótese

In [11]:
if acuracia_scaled > acuracia_sem_normalizar:
    print(f"A escala melhorou o modelo em {acuracia_scaled - acuracia_sem_normalizar:.2%}")
else:
    print("Os resultados foram próximos.")

Os resultados foram próximos.


### **O que foi feito?**

Nesta etapa final, realizamos uma estrutura condicional para checar se houve um ganho real de performace na padronização dos dados..